# Pocket-Agent — Complete Training & Inference Pipeline

**Single-file Colab notebook.** Upload only this file — no additional files required.

### Pipeline overview
| Section | Content |
|---|---|
| 0 | Configuration |
| 0b | Google Drive (optional) + artifact paths |
| 1 | Install dependencies |
| 2 | Tool schemas & prompt utilities |
| 3 | Synthetic training data generation |
| 4 | Model & tokenizer loading (QLoRA) |
| 5 | Fine-tuning with SFTTrainer |
| 6 | Evaluation harness |
| 7 | LoRA merge + int8 quantization |
| 8 | CPU latency benchmark |
| 9 | Write `inference.py` to disk |
| 10 | Gradio chatbot demo |

> **Tip:** Set `USE_GOOGLE_DRIVE=True` in the config cell (and `GOOGLE_DRIVE_ARTIFACTS` to a folder under your Drive) to persist `artifacts/` across Colab disconnects; run Section **0b** right after config.  
> **Tip:** Use **T4 GPU** runtime for training (`Runtime → Change runtime type → T4 GPU`). The demo (Section 10) runs on CPU after artifacts are saved.

## Section 0 — Configuration

In [7]:
# ============================================================
# CONFIGURATION — edit this cell before running
# ============================================================

# Base model (≤ 2B params). Qwen2.5-0.5B-Instruct: 494M, fast on CPU.
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Artifact storage (paths are finalized in the next cell after optional Drive mount)
# - USE_GOOGLE_DRIVE=False: artifacts under ./artifacts on the Colab VM (lost when runtime ends).
# - USE_GOOGLE_DRIVE=True: mount Drive, then store under GOOGLE_DRIVE_ARTIFACTS (persists across sessions).
USE_GOOGLE_DRIVE         = False
GOOGLE_DRIVE_ARTIFACTS   = "/content/drive/MyDrive/vyrothon"

# LoRA hyperparameters
LORA_RANK         = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODS  = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training hyperparameters
NUM_TRAIN_EPOCHS  = 1
BATCH_SIZE        = 4
GRAD_ACCUM        = 4           # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE     = 2e-4
MAX_SEQ_LENGTH    = 512
WARMUP_RATIO      = 0.03
VAL_SPLIT         = 0.1         # 10% of data for validation

# Dataset
EXAMPLES_PER_TOOL = 60          # single-turn examples per tool
PARAPHRASE_COUNT  = 40          # paraphrase examples
ADVERSARIAL_COUNT = 60          # adversarial / code-switched examples
REFUSAL_COUNT     = 80          # refusal examples
MULTITURN_COUNT   = 50          # multi-turn conversation examples
RANDOM_SEED       = 42

# Teacher LLM augmentation (optional)
# Set USE_TEACHER_LLM=True and pick TEACHER_BACKEND; set the matching API key env var.
USE_TEACHER_LLM   = False
TEACHER_BACKEND   = "groq"      # "openai" | "groq" | "anthropic" | "ollama"
TEACHER_MODEL     = "llama-3.3-70b-versatile"   # OpenAI / Groq chat model id
ANTHROPIC_MODEL   = "claude-3-5-haiku-20241022" # if TEACHER_BACKEND == "anthropic"
OLLAMA_MODEL      = "gemma4:e4b"                # if TEACHER_BACKEND == "ollama"
OLLAMA_BASE_URL   = "http://127.0.0.1:11434"    # local Ollama; use host.docker.internal etc. if needed
TEACHER_EXTRA     = 200         # cap / hint for extra synthetic variants (teacher cell uses 3 per seed)

# Inference generation params (keep short for speed)
MAX_NEW_TOKENS    = 128
DO_SAMPLE         = False

# Hugging Face token (required to download gated models like Llama)
HF_TOKEN = None  # e.g. "hf_xxxxx" — Qwen2.5 does NOT require a token

print("Configuration loaded (run next cell to set artifact paths / mount Drive).")


Configuration loaded (run next cell to set artifact paths / mount Drive).


In [8]:
# Section 0b — Google Drive (optional) + artifact directories
import os

# Finalize ARTIFACTS_DIR and subdirs (ADAPTER_DIR, QUANTIZED_DIR, TOKENIZER_DIR)
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print("Mounting Google Drive — approve access in the browser if prompted...")
        drive.mount("/content/drive")
        ARTIFACTS_DIR = GOOGLE_DRIVE_ARTIFACTS.rstrip("/")
    except ImportError:
        print("google.colab not available (local Jupyter). Falling back to ./artifacts")
        ARTIFACTS_DIR = "./artifacts"
else:
    ARTIFACTS_DIR = "./artifacts"

ADAPTER_DIR   = os.path.join(ARTIFACTS_DIR, "adapter")
QUANTIZED_DIR = os.path.join(ARTIFACTS_DIR, "quantized_model")
TOKENIZER_DIR = os.path.join(ARTIFACTS_DIR, "tokenizer")

for d in (ADAPTER_DIR, QUANTIZED_DIR, TOKENIZER_DIR):
    os.makedirs(d, exist_ok=True)

print(f"ARTIFACTS_DIR   = {ARTIFACTS_DIR!r}")
print(f"ADAPTER_DIR     = {ADAPTER_DIR!r}")
print(f"QUANTIZED_DIR   = {QUANTIZED_DIR!r}")
print(f"TOKENIZER_DIR   = {TOKENIZER_DIR!r}")


ARTIFACTS_DIR   = './artifacts'
ADAPTER_DIR     = './artifacts/adapter'
QUANTIZED_DIR   = './artifacts/quantized_model'
TOKENIZER_DIR   = './artifacts/tokenizer'


In [9]:
import os, torch

# Detect compute environment
HAS_GPU = torch.cuda.is_available()
DEVICE  = "cuda" if HAS_GPU else "cpu"
DTYPE   = torch.bfloat16 if HAS_GPU else torch.float32

if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("No GPU detected — running on CPU. Training will be skipped; demo will use pre-saved artifacts.")

print(f"Device: {DEVICE}, dtype: {DTYPE}")


No GPU detected — running on CPU. Training will be skipped; demo will use pre-saved artifacts.
Device: cpu, dtype: torch.float32


## Section 1 — Install Dependencies

In [10]:
# Install all required packages in one cell
# Pin ranges to avoid API-breaking updates during a 2-hour hackathon window
%pip install -q \
    "transformers>=5.0,<6.0" \
    "accelerate>=0.27" \
    "peft>=0.10" \
    "trl>=0.8" \
    "bitsandbytes>=0.43" \
    "datasets>=2.18" \
    "sentencepiece" \
    "protobuf" \
    "gradio>=4.0" \
    "safetensors" \
    "openai>=1.0" \
    "anthropic>=0.40"   # teacher: anthropic backend

print("All packages installed.")

All packages installed.


## Section 2 — Tool Schemas & Prompt Utilities

In [11]:
import json, re

# ---------------------------------------------------------------------------
# Exact tool schemas (embedded — no external file needed on Colab)
# ---------------------------------------------------------------------------
TOOL_SCHEMAS = [
    {"name": "weather",
     "description": "Get current weather for a location.",
     "parameters": {"type": "object",
                    "properties": {
                        "location": {"type": "string"},
                        "unit":     {"type": "string", "enum": ["C", "F"]}},
                    "required": ["location", "unit"]}},
    {"name": "calendar",
     "description": "List or create calendar events.",
     "parameters": {"type": "object",
                    "properties": {
                        "action": {"type": "string", "enum": ["list", "create"]},
                        "date":   {"type": "string"},
                        "title":  {"type": "string"}},
                    "required": ["action", "date"]}},
    {"name": "convert",
     "description": "Convert a numeric value between units.",
     "parameters": {"type": "object",
                    "properties": {
                        "value":     {"type": "number"},
                        "from_unit": {"type": "string"},
                        "to_unit":   {"type": "string"}},
                    "required": ["value", "from_unit", "to_unit"]}},
    {"name": "currency",
     "description": "Convert between currencies.",
     "parameters": {"type": "object",
                    "properties": {
                        "amount": {"type": "number"},
                        "from":   {"type": "string"},
                        "to":     {"type": "string"}},
                    "required": ["amount", "from", "to"]}},
    {"name": "sql",
     "description": "Execute a SQL query.",
     "parameters": {"type": "object",
                    "properties": {
                        "query": {"type": "string"}},
                    "required": ["query"]}},
]

VALID_TOOLS = {s["name"] for s in TOOL_SCHEMAS}

# ---------------------------------------------------------------------------
# System prompt (kept identical in training data, inference.py, and demo)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """You are Pocket-Agent, a concise on-device assistant with access to exactly five tools. \
When the user's request clearly matches a tool, respond ONLY with a JSON object wrapped in <tool_call>...</tool_call> tags. \
When no tool fits (chitchat, impossible requests, or ambiguous references with no conversation history), respond with plain natural language — no tool call.

Available tools and their required arguments:
  weather  : {{"location": "<city>", "unit": "C" or "F"}}
  calendar : {{"action": "list" or "create", "date": "YYYY-MM-DD", "title": "<string, create only>"}}
  convert  : {{"value": <number>, "from_unit": "<unit>", "to_unit": "<unit>"}}
  currency : {{"amount": <number>, "from": "<ISO3>", "to": "<ISO3>"}}
  sql      : {{"query": "<SQL string>"}}

Tool call format (emit exactly this, nothing else):
<tool_call>{{"tool": "tool_name", "args": {{...}}}}</tool_call>"""

# ---------------------------------------------------------------------------
# Helper utilities
# ---------------------------------------------------------------------------
def make_tool_call(tool: str, **args) -> str:
    """Construct a properly-formatted tool call string."""
    return f'<tool_call>{json.dumps({"tool": tool, "args": args}, ensure_ascii=False)}</tool_call>'


def parse_tool_call(text: str) -> dict | None:
    """Extract and parse JSON from <tool_call>...</tool_call> tags."""
    match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1).strip())
    except json.JSONDecodeError:
        return None


def is_refusal(text: str) -> bool:
    return parse_tool_call(text) is None


def format_messages(history: list[dict], user_utterance: str) -> list[dict]:
    """Build a messages list for apply_chat_template from history + new turn."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        if turn.get("role") in ("user", "assistant"):
            msgs.append({"role": turn["role"], "content": turn["content"]})
    msgs.append({"role": "user", "content": user_utterance})
    return msgs


# Verify helper
tc = make_tool_call("weather", location="Paris", unit="C")
assert parse_tool_call(tc) == {"tool": "weather", "args": {"location": "Paris", "unit": "C"}}
print("Tool utilities OK.")
print("Sample tool call:", tc)

Tool utilities OK.
Sample tool call: <tool_call>{"tool": "weather", "args": {"location": "Paris", "unit": "C"}}</tool_call>


## Section 3 — Synthetic Training Data Generation

In [13]:
import math
import random
from datetime import date, timedelta

random.seed(RANDOM_SEED)

# ---------------------------------------------------------------------------
# Pool definitions
# ---------------------------------------------------------------------------
CITIES_C = [
    "Paris", "London", "Berlin", "Tokyo", "Mumbai", "Delhi", "Beijing", "Seoul",
    "Sydney", "Cairo", "Lagos", "Nairobi", "Istanbul", "Tehran", "Karachi",
    "Bangkok", "Jakarta", "Manila", "Dhaka", "Singapore", "Kuala Lumpur",
    "Amsterdam", "Vienna", "Warsaw", "Prague", "Stockholm", "Oslo", "Helsinki",
    "Athens", "Rome", "Madrid", "Lisbon", "Brussels", "Zurich", "Dubai",
    "Riyadh", "Doha", "Casablanca", "Johannesburg", "Accra", "Addis Ababa",
]
CITIES_F = [
    "New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Philadelphia",
    "San Antonio", "San Diego", "Dallas", "San Jose", "Austin", "Jacksonville",
    "Miami", "Seattle", "Denver", "Nashville", "Boston", "Portland", "Las Vegas",
    "Atlanta", "Minneapolis", "Detroit", "Charlotte", "Orlando", "Tampa",
]

CURRENCY_PAIRS = [
    ("USD", "EUR"), ("EUR", "USD"), ("USD", "GBP"), ("GBP", "USD"),
    ("USD", "JPY"), ("JPY", "USD"), ("USD", "INR"), ("INR", "USD"),
    ("EUR", "GBP"), ("GBP", "EUR"), ("EUR", "JPY"), ("EUR", "CHF"),
    ("CHF", "USD"), ("CAD", "USD"), ("USD", "CAD"), ("AUD", "USD"),
    ("USD", "AUD"), ("GBP", "JPY"), ("CAD", "AUD"), ("USD", "CNY"),
    ("CNY", "USD"), ("USD", "MXN"), ("MXN", "USD"), ("USD", "BRL"),
    ("BRL", "USD"), ("EUR", "INR"), ("GBP", "INR"), ("USD", "PKR"),
    ("PKR", "USD"), ("INR", "PKR"), ("AED", "USD"), ("USD", "AED"),
    ("SAR", "USD"), ("USD", "SAR"), ("JPY", "INR"), ("KRW", "USD"),
    ("USD", "SGD"), ("SGD", "USD"), ("USD", "HKD"), ("HKD", "USD"),
]

UNIT_PAIRS = [
    ("kilometers", "miles"), ("miles", "kilometers"),
    ("kilograms", "pounds"), ("pounds", "kilograms"),
    ("Fahrenheit", "Celsius"), ("Celsius", "Fahrenheit"),
    ("liters", "gallons"), ("gallons", "liters"),
    ("meters", "feet"), ("feet", "meters"),
    ("centimeters", "inches"), ("inches", "centimeters"),
    ("grams", "ounces"), ("ounces", "grams"),
    ("kilograms", "grams"), ("grams", "kilograms"),
    ("miles per hour", "kilometers per hour"), ("kilometers per hour", "miles per hour"),
    ("square meters", "square feet"), ("square feet", "square meters"),
    ("hectares", "acres"), ("acres", "hectares"),
    ("megabytes", "gigabytes"), ("gigabytes", "megabytes"),
    ("seconds", "minutes"), ("minutes", "hours"),
    ("watts", "kilowatts"), ("horsepower", "kilowatts"),
]

SQL_TEMPLATES = [
    ("Show all users", "SELECT * FROM users"),
    ("Get all active products", "SELECT * FROM products WHERE active = 1"),
    ("Count orders", "SELECT COUNT(*) FROM orders"),
    ("Find pending orders", "SELECT * FROM orders WHERE status = 'pending'"),
    ("Get recent transactions", "SELECT * FROM transactions ORDER BY created_at DESC LIMIT 10"),
    ("List all tables", "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"),
    ("Delete expired sessions", "DELETE FROM sessions WHERE expires_at < NOW()"),
    ("Update user email", "UPDATE users SET email = 'new@example.com' WHERE id = 1"),
    ("Insert new log", "INSERT INTO logs (event, timestamp) VALUES ('login', NOW())"),
    ("Get top customers", "SELECT customer_id, SUM(amount) as total FROM orders GROUP BY customer_id ORDER BY total DESC LIMIT 5"),
    ("Find users without orders", "SELECT u.id, u.name FROM users u LEFT JOIN orders o ON u.id = o.user_id WHERE o.id IS NULL"),
    ("Get product inventory", "SELECT name, stock FROM products WHERE stock < 10"),
    ("Archive old records", "INSERT INTO archive SELECT * FROM events WHERE created_at < '2023-01-01'"),
    ("Get user by email", "SELECT * FROM users WHERE email = 'user@example.com'"),
    ("Count active sessions", "SELECT COUNT(*) FROM sessions WHERE expires_at > NOW()"),
    ("Get monthly revenue", "SELECT DATE_TRUNC('month', created_at) as month, SUM(amount) FROM orders GROUP BY month"),
]

CALENDAR_TITLES = [
    "Team Standup", "Doctor Appointment", "Project Review", "Birthday Party",
    "Weekly Sync", "Board Meeting", "Dentist Appointment", "Gym Session",
    "Lunch with Sarah", "Flight to London", "Hotel Checkout", "Conference Call",
    "Code Review", "Product Demo", "Sprint Planning", "Retrospective",
    "Interview", "Yoga Class", "Car Service", "Tax Filing Deadline",
    "Annual Review", "Quarterly Report", "Team Dinner", "Client Meeting",
]


def random_date() -> str:
    base = date(2024, 1, 1)
    offset = random.randint(0, 730)
    return (base + timedelta(days=offset)).strftime("%Y-%m-%d")


def random_amount(low=1, high=10000) -> float:
    if random.random() < 0.6:
        lo_i = math.ceil(low)
        hi_i = math.floor(high)
        if lo_i > hi_i:
            return round(random.uniform(low, high), 2)
        return float(random.randint(lo_i, hi_i))
    return round(random.uniform(low, high), 2)


# ---------------------------------------------------------------------------
# Phrasing templates
# ---------------------------------------------------------------------------
WEATHER_PROMPTS = [
    "What's the weather in {city}?",
    "What is the current temperature in {city} in {unit_word}?",
    "Tell me the weather in {city}.",
    "How's the weather in {city} right now?",
    "What's the temperature in {city} in {unit_sym}?",
    "Is it hot or cold in {city} today?",
    "Give me the weather for {city}.",
    "I need the current weather in {city}.",
    "What are the current conditions in {city}?",
    "Check the weather in {city} for me.",
    "Weather update for {city} please.",
    "Can you get me the weather for {city}?",
    "What's the forecast for {city} in {unit_word}?",
    "I'm planning to go to {city} — what's the weather?",
]

CALENDAR_LIST_PROMPTS = [
    "Show me my calendar for {date}.",
    "What events do I have on {date}?",
    "List my schedule for {date}.",
    "What's on my calendar on {date}?",
    "Check my appointments on {date}.",
    "Do I have anything scheduled on {date}?",
    "What meetings do I have on {date}?",
    "Pull up my calendar for {date}.",
]

CALENDAR_CREATE_PROMPTS = [
    "Create a calendar event called {title} on {date}.",
    "Add {title} to my calendar on {date}.",
    "Schedule a {title} event on {date}.",
    "Book {title} on {date} in my calendar.",
    "Put {title} on my calendar for {date}.",
    "Set up a {title} on {date}.",
    "I need to add {title} to {date} in my calendar.",
    "Remind me about {title} on {date}.",
    "Create an event: {title} on {date}.",
]

CONVERT_PROMPTS = [
    "Convert {value} {from_unit} to {to_unit}.",
    "How many {to_unit} is {value} {from_unit}?",
    "What is {value} {from_unit} in {to_unit}?",
    "I need to convert {value} {from_unit} to {to_unit}.",
    "{value} {from_unit} in {to_unit} please.",
    "Change {value} {from_unit} to {to_unit}.",
    "What's {value} {from_unit} expressed in {to_unit}?",
]

CURRENCY_PROMPTS = [
    "Convert {amount} {from_cur} to {to_cur}.",
    "How much is {amount} {from_cur} in {to_cur}?",
    "What is {amount} {from_cur} in {to_cur}?",
    "Exchange {amount} {from_cur} for {to_cur}.",
    "I want to convert {amount} {from_cur} into {to_cur}.",
    "How many {to_cur} do I get for {amount} {from_cur}?",
    "What's the equivalent of {amount} {from_cur} in {to_cur}?",
    "{amount} {from_cur} to {to_cur} please.",
]

SQL_PROMPTS = [
    "Run this query: {query}",
    "Execute: {query}",
    "Run the following SQL: {query}",
    "{intent}. SQL: {query}",
    "Please run: {query}",
    "Execute this SQL statement: {query}",
    "Query the database: {query}",
]

REFUSAL_PROMPTS = [
    "How are you doing today?",
    "Tell me a joke.",
    "What's the meaning of life?",
    "Can you write me a poem?",
    "Order me a pizza.",
    "Book me a flight to Tokyo.",
    "Play some music for me.",
    "Turn off the lights.",
    "Send an email to my boss.",
    "Call my mom.",
    "Set an alarm for 7 AM.",
    "What is 2 + 2?",
    "Write me a cover letter.",
    "Translate this to French: Hello.",
    "Find me a restaurant nearby.",
    "What's the news today?",
    "Search the web for cats.",
    "Who won the World Cup?",
    "Generate an image of a sunset.",
    "Open Spotify.",
    "Take a screenshot.",
    "What's 15% of 240?",
    "Tell me about quantum physics.",
    "What's the best programming language?",
    "Who is the president?",
    "Set a timer for 5 minutes.",
    "Lock my phone.",
    "Navigate to the nearest coffee shop.",
    "Check my bank balance.",
    "Post this to Twitter: Hello world!",
    "Can you hack into my ex's account?",
    "Tell me the lottery numbers.",
    "What will the stock market do tomorrow?",
    "Am I going to win the game today?",
    "Compose a haiku about autumn.",
    "Give me investment advice.",
    "What medication should I take?",
    "Diagnose my illness.",
    "Write my essay for school.",
    "Do my homework.",
]

REFUSAL_RESPONSES = [
    "I'm not able to help with that, but I can assist with weather, calendar events, unit conversions, currency exchange, and database queries!",
    "That's outside my capabilities. I have tools for weather, calendar, unit conversion, currency, and SQL queries.",
    "I don't have a tool for that. I can help with weather, calendar, conversions, currency exchange, and SQL.",
    "Sorry, I can't do that. My available tools are: weather, calendar, unit conversion, currency, and SQL.",
    "I'm afraid that's beyond what I can do. Feel free to ask about weather, dates, unit conversions, currencies, or database queries!",
]


# ---------------------------------------------------------------------------
# Adversarial / code-switched templates
# ---------------------------------------------------------------------------
ADVERSARIAL_WEATHER = [
    ("wht's da wether in {city}?", "{city}", "C"),
    ("tempature in {city} pls", "{city}", "C"),
    ("wheather for {city}?", "{city}", "C"),
    ("मुझे {city} का मौसम बताओ", "{city}", "C"),
    ("{city} mein mausam kaisa hai? Celsius mein", "{city}", "C"),
    ("موسم کیسا ہے {city} میں؟", "{city}", "C"),
    ("¿Qué tiempo hace en {city}? En Celsius.", "{city}", "C"),
    ("كم درجة الحرارة في {city}؟ بالسيلزيوس", "{city}", "C"),
    ("Che tempo fa a {city}?", "{city}", "C"),
    ("What's da temp in {city} in F?", "{city}", "F"),
    ("temprature in {city} Farenhiet?", "{city}", "F"),
    ("{city} ka mausam batao F mein", "{city}", "F"),
    ("weather 4 {city}?", "{city}", "C"),
    ("i need wether update {city}", "{city}", "C"),
]

ADVERSARIAL_CURRENCY = [
    ("convrt {amount} {from_cur} to {to_cur}", "{amount}", "{from_cur}", "{to_cur}"),
    ("{amount} {from_cur} mein {to_cur} kitne hain?", "{amount}", "{from_cur}", "{to_cur}"),
    ("¿Cuánto es {amount} {from_cur} en {to_cur}?", "{amount}", "{from_cur}", "{to_cur}"),
    ("{amount} {from_cur} ko {to_cur} mein convert karo", "{amount}", "{from_cur}", "{to_cur}"),
    ("{amount} {from_cur} يساوي كم {to_cur}؟", "{amount}", "{from_cur}", "{to_cur}"),
]

ADVERSARIAL_CONVERT = [
    ("convrt {value} {from_unit} to {to_unit}", "{value}", "{from_unit}", "{to_unit}"),
    ("{value} {from_unit} ko {to_unit} mein badlo", "{value}", "{from_unit}", "{to_unit}"),
    ("¿Cuántos {to_unit} son {value} {from_unit}?", "{value}", "{from_unit}", "{to_unit}"),
    ("{value} {from_unit} = how many {to_unit}?", "{value}", "{from_unit}", "{to_unit}"),
]

REFUSAL_AMBIGUOUS = [
    "Convert it.",
    "What's the weather?",
    "Show me my calendar.",
    "Run the query.",
    "Convert that.",
    "Check the exchange rate.",
    "What's it in the other unit?",
    "Run it.",
]

REFUSAL_AMBIGUOUS_RESPONSES = [
    "Could you clarify what you'd like to convert? I need a value, source unit, and target unit.",
    "Which city would you like the weather for? Please also specify Celsius or Fahrenheit.",
    "Could you specify a date so I can list your calendar events?",
    "Could you provide the SQL query you'd like me to run?",
    "Could you clarify what value and units you'd like to convert?",
    "Could you specify an amount and the currencies you want to exchange?",
    "I need more context — what value and units should I convert?",
    "Please provide more details about what you'd like me to run.",
]


# ---------------------------------------------------------------------------
# Multi-turn templates
# ---------------------------------------------------------------------------
def make_multiturn_currency_example():
    amount = random_amount(10, 5000)
    from_cur, to_cur = random.choice(CURRENCY_PAIRS)
    history = [
        {"role": "user", "content": f"I have {amount} {from_cur}."},
        {"role": "assistant", "content": f"Got it — {amount} {from_cur}. Would you like to convert it to another currency?"},
    ]
    user_turn = f"Yes, convert that to {to_cur}."
    assistant_turn = make_tool_call("currency", amount=amount, **{"from": from_cur, "to": to_cur})
    return history, user_turn, assistant_turn


def make_multiturn_convert_example():
    value = random_amount(0.1, 1000)
    from_unit, to_unit = random.choice(UNIT_PAIRS)
    history = [
        {"role": "user", "content": f"The measurement is {value} {from_unit}."},
        {"role": "assistant", "content": f"Noted — {value} {from_unit}. Want me to convert that to {to_unit}?"},
    ]
    user_turn = "Yes please."
    assistant_turn = make_tool_call("convert", value=value, from_unit=from_unit, to_unit=to_unit)
    return history, user_turn, assistant_turn


def make_multiturn_calendar_example():
    d = random_date()
    title = random.choice(CALENDAR_TITLES)
    history = [
        {"role": "user", "content": f"I need to add something to my calendar."},
        {"role": "assistant", "content": "Sure! What event would you like to add and on which date?"},
        {"role": "user", "content": f"It's called {title} and it's on {d}."},
        {"role": "assistant", "content": f"Got it — adding {title} on {d}. Shall I go ahead?"},
    ]
    user_turn = "Yes, create it."
    assistant_turn = make_tool_call("calendar", action="create", date=d, title=title)
    return history, user_turn, assistant_turn


def make_multiturn_sql_example():
    intent, query = random.choice(SQL_TEMPLATES)
    history = [
        {"role": "user", "content": "I need to query the database."},
        {"role": "assistant", "content": "Sure, what SQL query would you like me to run?"},
    ]
    user_turn = query
    assistant_turn = make_tool_call("sql", query=query)
    return history, user_turn, assistant_turn


# ---------------------------------------------------------------------------
# Build training examples
# ---------------------------------------------------------------------------
raw_examples = []   # list of (history, user_turn, assistant_turn, slice_label)

# Slice A: single-turn, in-distribution
for _ in range(EXAMPLES_PER_TOOL):
    city = random.choice(CITIES_C)
    tmpl = random.choice(WEATHER_PROMPTS)
    prompt = tmpl.format(city=city, unit_word="Celsius", unit_sym="C")
    raw_examples.append(([], prompt, make_tool_call("weather", location=city, unit="C"), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    city = random.choice(CITIES_F)
    tmpl = random.choice(WEATHER_PROMPTS)
    prompt = tmpl.format(city=city, unit_word="Fahrenheit", unit_sym="F")
    raw_examples.append(([], prompt, make_tool_call("weather", location=city, unit="F"), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    d = random_date()
    tmpl = random.choice(CALENDAR_LIST_PROMPTS)
    prompt = tmpl.format(date=d)
    raw_examples.append(([], prompt, make_tool_call("calendar", action="list", date=d), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    d = random_date()
    title = random.choice(CALENDAR_TITLES)
    tmpl = random.choice(CALENDAR_CREATE_PROMPTS)
    prompt = tmpl.format(date=d, title=title)
    raw_examples.append(([], prompt, make_tool_call("calendar", action="create", date=d, title=title), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    value = random_amount(0.1, 1000)
    from_unit, to_unit = random.choice(UNIT_PAIRS)
    tmpl = random.choice(CONVERT_PROMPTS)
    prompt = tmpl.format(value=value, from_unit=from_unit, to_unit=to_unit)
    raw_examples.append(([], prompt, make_tool_call("convert", value=value, from_unit=from_unit, to_unit=to_unit), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    amount = random_amount(1, 10000)
    from_cur, to_cur = random.choice(CURRENCY_PAIRS)
    tmpl = random.choice(CURRENCY_PROMPTS)
    prompt = tmpl.format(amount=amount, from_cur=from_cur, to_cur=to_cur)
    raw_examples.append(([], prompt, make_tool_call("currency", amount=amount, **{"from": from_cur, "to": to_cur}), "A"))

for _ in range(EXAMPLES_PER_TOOL):
    intent, query = random.choice(SQL_TEMPLATES)
    tmpl = random.choice(SQL_PROMPTS)
    prompt = tmpl.format(query=query, intent=intent)
    raw_examples.append(([], prompt, make_tool_call("sql", query=query), "A"))

# Slice B: paraphrased (alternate surface forms)
PARAPHRASE_LEADS = [
    "Could you tell me", "I'd like to know", "Can you check",
    "Please look up", "I was wondering", "Quickly check",
]
for _ in range(PARAPHRASE_COUNT):
    city = random.choice(CITIES_C + CITIES_F)
    unit = "C" if city in CITIES_C else "F"
    lead = random.choice(PARAPHRASE_LEADS)
    prompt = f"{lead} the weather in {city}? Give me {unit}."
    raw_examples.append(([], prompt, make_tool_call("weather", location=city, unit=unit), "B"))

for _ in range(PARAPHRASE_COUNT // 2):
    amount = random_amount()
    from_cur, to_cur = random.choice(CURRENCY_PAIRS)
    lead = random.choice(PARAPHRASE_LEADS)
    prompt = f"{lead} the exchange of {amount} {from_cur} to {to_cur}."
    raw_examples.append(([], prompt, make_tool_call("currency", amount=amount, **{"from": from_cur, "to": to_cur}), "B"))

for _ in range(PARAPHRASE_COUNT // 2):
    value = random_amount(0.5, 500)
    from_unit, to_unit = random.choice(UNIT_PAIRS)
    lead = random.choice(PARAPHRASE_LEADS)
    prompt = f"{lead} how many {to_unit} are in {value} {from_unit}."
    raw_examples.append(([], prompt, make_tool_call("convert", value=value, from_unit=from_unit, to_unit=to_unit), "B"))

# Slice C: adversarial
for _ in range(ADVERSARIAL_COUNT // 2):
    tmpl, *_ = random.choice(ADVERSARIAL_WEATHER)
    city = random.choice(CITIES_C)
    unit = "F" if "F" in tmpl or "Farenhiet" in tmpl else "C"
    prompt = tmpl.replace("{city}", city)
    raw_examples.append(([], prompt, make_tool_call("weather", location=city, unit=unit), "C"))

for _ in range(ADVERSARIAL_COUNT // 4):
    tmpl = random.choice(ADVERSARIAL_CURRENCY)
    amount = random_amount()
    from_cur, to_cur = random.choice(CURRENCY_PAIRS)
    prompt = tmpl[0].replace("{amount}", str(amount)).replace("{from_cur}", from_cur).replace("{to_cur}", to_cur)
    raw_examples.append(([], prompt, make_tool_call("currency", amount=amount, **{"from": from_cur, "to": to_cur}), "C"))

for _ in range(ADVERSARIAL_COUNT // 4):
    tmpl = random.choice(ADVERSARIAL_CONVERT)
    value = random_amount()
    from_unit, to_unit = random.choice(UNIT_PAIRS)
    prompt = tmpl[0].replace("{value}", str(value)).replace("{from_unit}", from_unit).replace("{to_unit}", to_unit)
    raw_examples.append(([], prompt, make_tool_call("convert", value=value, from_unit=from_unit, to_unit=to_unit), "C"))

# Slice D: refusals + ambiguous
for i, (prompt, response) in enumerate(zip(REFUSAL_PROMPTS, REFUSAL_RESPONSES * 2)):
    raw_examples.append(([], prompt, response, "D"))

for prompt, response in zip(REFUSAL_AMBIGUOUS, REFUSAL_AMBIGUOUS_RESPONSES):
    raw_examples.append(([], prompt, response, "D"))

# Additional refusals
for _ in range(REFUSAL_COUNT - len(REFUSAL_PROMPTS) - len(REFUSAL_AMBIGUOUS)):
    prompt = random.choice(REFUSAL_PROMPTS)
    response = random.choice(REFUSAL_RESPONSES)
    raw_examples.append(([], prompt, response, "D"))

# Multi-turn (Slice D)
for _ in range(MULTITURN_COUNT // 4):
    h, u, a = make_multiturn_currency_example()
    raw_examples.append((h, u, a, "D"))

for _ in range(MULTITURN_COUNT // 4):
    h, u, a = make_multiturn_convert_example()
    raw_examples.append((h, u, a, "D"))

for _ in range(MULTITURN_COUNT // 4):
    h, u, a = make_multiturn_calendar_example()
    raw_examples.append((h, u, a, "D"))

for _ in range(MULTITURN_COUNT // 4):
    h, u, a = make_multiturn_sql_example()
    raw_examples.append((h, u, a, "D"))

random.shuffle(raw_examples)
print(f"Generated {len(raw_examples)} raw training examples.")
slice_counts = {}
for _, _, _, sl in raw_examples:
    slice_counts[sl] = slice_counts.get(sl, 0) + 1
print("Slice distribution:", slice_counts)

Generated 658 raw training examples.
Slice distribution: {'B': 80, 'A': 420, 'D': 98, 'C': 60}


In [14]:
# ---------------------------------------------------------------------------
# Optional: Teacher-LLM augmentation (USE_TEACHER_LLM=True)
# Backends: openai (OPENAI_API_KEY), groq (GROQ_API_KEY), anthropic (ANTHROPIC_API_KEY), ollama (local)
# ---------------------------------------------------------------------------
import os
import json as _json
import re as _re


def _teacher_messages(seed_str: str):
    sys = """You are generating training examples for a tool-calling assistant.
Given a seed example, produce 3 paraphrased or adversarial variants.
Each variant must be a JSON object with keys: 'user' (the user prompt) and 'assistant' (the correct response).
Adversarial variants may include typos, mixed-language input, unit ambiguity, or hallucination-bait names.
Respond ONLY with a JSON array of objects — no markdown fences."""
    user = f"Seed: {seed_str}\nGenerate exactly 3 variants as a JSON array."
    return sys, user


def _parse_json_array(text: str):
    text = (text or "").strip()
    m = _re.search(r"\[[\s\S]*\]", text)
    if not m:
        raise ValueError("No JSON array in teacher response")
    return _json.loads(m.group(0))


def _teacher_complete_openai_compat(base_url, api_key, model_id, seed_str):
    from openai import OpenAI
    kwargs = {"api_key": api_key}
    if base_url:
        kwargs["base_url"] = base_url
    client = OpenAI(**kwargs)
    sys, user = _teacher_messages(seed_str)
    resp = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "system", "content": sys}, {"role": "user", "content": user}],
        max_tokens=512,
        temperature=0.9,
    )
    return resp.choices[0].message.content


def _teacher_complete_anthropic(seed_str):
    import anthropic
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        raise RuntimeError("ANTHROPIC_API_KEY not set")
    sys, user = _teacher_messages(seed_str)
    client = anthropic.Anthropic(api_key=key)
    resp = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        system=sys,
        messages=[{"role": "user", "content": user}],
    )
    return resp.content[0].text


def _teacher_complete_ollama(seed_str):
    import urllib.request
    sys, user = _teacher_messages(seed_str)
    url = OLLAMA_BASE_URL.rstrip("/") + "/api/chat"
    body = _json.dumps(
        {
            "model": OLLAMA_MODEL,
            "messages": [{"role": "system", "content": sys}, {"role": "user", "content": user}],
            "stream": False,
        }
    ).encode("utf-8")
    req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=120) as r:
        data = _json.loads(r.read().decode("utf-8"))
    return data["message"]["content"]


def _teacher_complete(seed_str: str) -> str:
    b = (TEACHER_BACKEND or "openai").lower()
    if b == "openai":
        key = os.environ.get("OPENAI_API_KEY", "")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set")
        return _teacher_complete_openai_compat(None, key, TEACHER_MODEL, seed_str)
    if b == "groq":
        key = os.environ.get("GROQ_API_KEY", "")
        if not key:
            raise RuntimeError("GROQ_API_KEY not set")
        return _teacher_complete_openai_compat("https://api.groq.com/openai/v1", key, TEACHER_MODEL, seed_str)
    if b == "anthropic":
        return _teacher_complete_anthropic(seed_str)
    if b == "ollama":
        return _teacher_complete_ollama(seed_str)
    raise ValueError(f"Unknown TEACHER_BACKEND: {TEACHER_BACKEND!r}")


if USE_TEACHER_LLM:
    teacher_extras = []
    try:
        seed_sample = random.sample(raw_examples, min(20, len(raw_examples)))
        for h, u, a, sl in seed_sample[:10]:
            try:
                seed_str = json.dumps({"user": u, "assistant": a})
                raw = _teacher_complete(seed_str)
                variants = _parse_json_array(raw)
                for v in variants:
                    if isinstance(v, dict) and "user" in v and "assistant" in v:
                        teacher_extras.append(([], v["user"], v["assistant"], "C"))
            except Exception as e:
                print(f"Teacher API error: {e}")
        raw_examples.extend(teacher_extras)
        print(f"Added {len(teacher_extras)} teacher-augmented examples. Total: {len(raw_examples)}.")
    except Exception as e:
        print(f"Teacher augmentation skipped: {e}")
else:
    print("Teacher-LLM augmentation disabled (USE_TEACHER_LLM = False). Using offline data only.")


Teacher-LLM augmentation disabled (USE_TEACHER_LLM = False). Using offline data only.


In [15]:
# ---------------------------------------------------------------------------
# Assemble HuggingFace Dataset
# We format each example using the tokenizer's chat template at training time.
# This cell just stores raw dicts; formatting happens after tokenizer is loaded.
# ---------------------------------------------------------------------------
dataset_records = []
for history, user_turn, assistant_turn, slice_label in raw_examples:
    dataset_records.append({
        "history":        history,
        "user_turn":      user_turn,
        "assistant_turn": assistant_turn,
        "slice":          slice_label,
    })

# Train/val split
random.shuffle(dataset_records)
val_size = max(20, int(len(dataset_records) * VAL_SPLIT))
val_records   = dataset_records[:val_size]
train_records = dataset_records[val_size:]

print(f"Train examples: {len(train_records)}")
print(f"Val   examples: {len(val_records)}")
print("Sample train record:")
sample = train_records[0]
print(f"  Slice: {sample['slice']}")
print(f"  User:  {sample['user_turn'][:80]}")
print(f"  Asst:  {sample['assistant_turn'][:80]}")

Train examples: 593
Val   examples: 65
Sample train record:
  Slice: A
  User:  How many square feet is 118.0 square meters?
  Asst:  <tool_call>{"tool": "convert", "args": {"value": 118.0, "from_unit": "square met


## Section 4 — Model & Tokenizer Loading (QLoRA)

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ---------------------------------------------------------------------------
# Tokenizer
# ---------------------------------------------------------------------------
print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    token=HF_TOKEN,
)

# Ensure pad token is set (Qwen2.5 uses <|endoftext|> as both EOS and PAD)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-pad for causal LM training

print(f"Vocab size:   {tokenizer.vocab_size}")
print(f"EOS token:    {tokenizer.eos_token!r}")
print(f"PAD token:    {tokenizer.pad_token!r}")

# Save tokenizer to artifacts
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"Tokenizer saved to {TOKENIZER_DIR}")

Loading tokenizer: Qwen/Qwen2.5-0.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size:   151643
EOS token:    '<|im_end|>'
PAD token:    '<|endoftext|>'
Tokenizer saved to ./artifacts/tokenizer


In [17]:
# ---------------------------------------------------------------------------
# Format raw records into text using the chat template
# ---------------------------------------------------------------------------
from datasets import Dataset


def record_to_text(record: dict) -> str:
    """Apply the chat template to produce a full training string."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in record["history"]:
        msgs.append({"role": turn["role"], "content": turn["content"]})
    msgs.append({"role": "user",      "content": record["user_turn"]})
    msgs.append({"role": "assistant", "content": record["assistant_turn"]})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)


train_texts = [{"text": record_to_text(r)} for r in train_records]
val_texts   = [{"text": record_to_text(r)} for r in val_records]

train_dataset = Dataset.from_list(train_texts)
val_dataset   = Dataset.from_list(val_texts)

print(f"Train dataset: {train_dataset}")
print(f"Val   dataset: {val_dataset}")
print("\nSample training text (truncated):")
print(train_dataset[0]["text"][:400])

Train dataset: Dataset({
    features: ['text'],
    num_rows: 593
})
Val   dataset: Dataset({
    features: ['text'],
    num_rows: 65
})

Sample training text (truncated):
<|im_start|>system
You are Pocket-Agent, a concise on-device assistant with access to exactly five tools. When the user's request clearly matches a tool, respond ONLY with a JSON object wrapped in <tool_call>...</tool_call> tags. When no tool fits (chitchat, impossible requests, or ambiguous references with no conversation history), respond with plain natural language — no tool call.

Available to


In [18]:
if not HAS_GPU:
    print("No GPU detected — skipping model load for training. Jump to Section 7 to load artifacts.")
else:
    print(f"Loading base model: {BASE_MODEL} with 4-bit QLoRA...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

    # Required before applying LoRA to a quantized model
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False   # disable KV cache during training

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODS,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print("Model ready for training.")

No GPU detected — skipping model load for training. Jump to Section 7 to load artifacts.


## Section 5 — Fine-tuning with SFTTrainer

In [19]:
if not HAS_GPU:
    print("Skipping training (no GPU). Load pre-trained artifacts in Section 7.")
else:
    from trl import SFTConfig, SFTTrainer

    sft_config = SFTConfig(
        # Output
        output_dir=ADAPTER_DIR,
        # Training schedule
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        learning_rate=LEARNING_RATE,
        # Precision
        bf16=True,
        # Sequence
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        # Evaluation & checkpointing
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        # Logging
        logging_steps=10,
        report_to="none",
        # Optimizer
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        # Packing (slightly faster for short sequences)
        packing=False,
        # Reproducibility
        seed=RANDOM_SEED,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        # peft_config is already applied above via get_peft_model
    )

    print("Starting training...")
    train_result = trainer.train()

    print("\nTraining complete!")
    print(f"  Train loss: {train_result.training_loss:.4f}")

    # Save the LoRA adapter
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"  LoRA adapter saved to: {ADAPTER_DIR}")

Skipping training (no GPU). Load pre-trained artifacts in Section 7.


## Section 6 — Evaluation Harness

In [20]:
# ---------------------------------------------------------------------------
# Embedded evaluation logic (matches eval_harness_contract.py exactly)
# ---------------------------------------------------------------------------

def args_match(pred_args: dict, gold_args: dict, tol: float = 0.01) -> bool:
    if set(pred_args.keys()) != set(gold_args.keys()):
        return False
    for key in gold_args:
        pv, gv = pred_args.get(key), gold_args[key]
        if isinstance(gv, (int, float)):
            try:
                if abs(float(pv) - float(gv)) / (abs(float(gv)) + 1e-9) > tol:
                    return False
            except (ValueError, TypeError):
                return False
        else:
            if str(pv).strip() != str(gv).strip():
                return False
    return True


def score_single(prediction: str, ground_truth: dict) -> float:
    expected_refusal = ground_truth.get("is_refusal", False)
    pred_call = parse_tool_call(prediction)
    pred_is_refusal = pred_call is None

    if expected_refusal:
        return 1.0 if pred_is_refusal else -0.5

    if pred_is_refusal:
        return 0.0

    expected_tool = ground_truth.get("expected_tool")
    expected_args = ground_truth.get("expected_args", {})

    if pred_call.get("tool") != expected_tool:
        return 0.0

    pred_args = pred_call.get("args", {})
    return 1.0 if args_match(pred_args, expected_args) else 0.5


print("Evaluation utilities loaded.")
# Quick sanity check
assert score_single(make_tool_call("weather", location="Paris", unit="C"),
                    {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Paris", "unit": "C"}}) == 1.0
assert score_single("No tool needed!", {"is_refusal": True}) == 1.0
assert score_single(make_tool_call("weather", location="Paris", unit="C"), {"is_refusal": True}) == -0.5
print("All scoring sanity checks passed.")

Evaluation utilities loaded.
All scoring sanity checks passed.


In [30]:
# ---------------------------------------------------------------------------
# Inference function (used for eval; mirrors inference.py exactly)
# ---------------------------------------------------------------------------
_eval_model     = None
_eval_tokenizer = None
_eval_device    = DEVICE


def _ensure_eval_model():
    global _eval_model, _eval_tokenizer
    if _eval_model is None:
        raise RuntimeError(
            "_eval_model is not loaded. "
            "Run Section 5 (training) or Section 7 (load quantized) first."
        )


def set_eval_model(m, tok):
    """Register the model and tokenizer to use for evaluation."""
    global _eval_model, _eval_tokenizer
    _eval_model     = m
    _eval_tokenizer = tok


def run(prompt: str, history: list[dict]) -> str:
    """Generate a response. Matches the grader-contract signature exactly."""
    _ensure_eval_model()
    msgs = format_messages(history, prompt)
    # Transformers v5: apply_chat_template returns BatchEncoding — use .to() then generate(**batch)
    enc = _eval_tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    )
    if isinstance(enc, torch.Tensor):
        enc = enc.to(_eval_device)
        prompt_len = enc.shape[1]
        with torch.no_grad():
            out = _eval_model.generate(
                enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                temperature=None,
                top_p=None,
                pad_token_id=_eval_tokenizer.eos_token_id,
                eos_token_id=_eval_tokenizer.eos_token_id,
            )
    else:
        enc = enc.to(_eval_device)
        prompt_len = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = _eval_model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=DO_SAMPLE,
                temperature=None,
                top_p=None,
                pad_token_id=_eval_tokenizer.eos_token_id,
                eos_token_id=_eval_tokenizer.eos_token_id,
            )
    new_tokens = out[0, prompt_len:]
    return _eval_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("run() function defined (requires model to be loaded via set_eval_model()).")

run() function defined (requires model to be loaded via set_eval_model()).


In [31]:
# ---------------------------------------------------------------------------
# Register the fine-tuned model for evaluation (skip if no GPU / no model)
# ---------------------------------------------------------------------------
if HAS_GPU and 'model' in dir() and model is not None:
    model.config.use_cache = True   # re-enable for inference
    model.eval()
    set_eval_model(model, tokenizer)

    # Evaluate on validation slice
    print("Evaluating on validation set...")
    scores = []
    for record in val_records[:30]:  # cap at 30 to save time
        pred = run(record["user_turn"], record["history"])
        # Build ground truth from the record
        pred_call = parse_tool_call(record["assistant_turn"])
        if pred_call is None:
            gt = {"is_refusal": True}
        else:
            gt = {
                "is_refusal": False,
                "expected_tool": pred_call["tool"],
                "expected_args": pred_call.get("args", {}),
            }
        s = score_single(pred, gt)
        scores.append(s)

    avg = sum(scores) / len(scores)
    print(f"\nValidation score ({len(scores)} examples): {avg:.3f}")
    print(f"  +1.0: {scores.count(1.0)}, +0.5: {scores.count(0.5)}, 0.0: {scores.count(0.0)}, -0.5: {scores.count(-0.5)}")
else:
    print("No model loaded — skipping validation eval.")

No model loaded — skipping validation eval.


In [32]:
# ---------------------------------------------------------------------------
# Evaluate on the embedded 40-example public dev set
# ---------------------------------------------------------------------------
PUBLIC_DEV_JSONL = """{
\"id\": \"dev_001\", \"slice\": \"A\",
\"conversation\": [{\"role\": \"user\", \"content\": \"What is the weather in Sydney?\"}],
\"ground_truth\": {\"is_refusal\": false, \"expected_tool\": \"weather\", \"expected_args\": {\"location\": \"Sydney\", \"unit\": \"C\"}}
}"""

# Embedded dev set records (same as starter/public_test.jsonl)
DEV_EXAMPLES = [
    {"id": "dev_001", "slice": "A", "conversation": [{"role": "user", "content": "What is the weather in Sydney?"}], "ground_truth": {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Sydney", "unit": "C"}}},
    {"id": "dev_002", "slice": "A", "conversation": [{"role": "user", "content": "Tell me the temperature in Miami in Fahrenheit"}], "ground_truth": {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Miami", "unit": "F"}}},
    {"id": "dev_003", "slice": "A", "conversation": [{"role": "user", "content": "List my calendar for 2024-07-04"}], "ground_truth": {"is_refusal": False, "expected_tool": "calendar", "expected_args": {"action": "list", "date": "2024-07-04"}}},
    {"id": "dev_004", "slice": "A", "conversation": [{"role": "user", "content": "Add Doctor Appointment to my calendar on 2024-05-20"}], "ground_truth": {"is_refusal": False, "expected_tool": "calendar", "expected_args": {"action": "create", "date": "2024-05-20", "title": "Doctor Appointment"}}},
    {"id": "dev_005", "slice": "A", "conversation": [{"role": "user", "content": "Convert 32 Fahrenheit to Celsius"}], "ground_truth": {"is_refusal": False, "expected_tool": "convert", "expected_args": {"value": 32, "from_unit": "Fahrenheit", "to_unit": "Celsius"}}},
    {"id": "dev_006", "slice": "A", "conversation": [{"role": "user", "content": "How much is 1500 GBP in JPY?"}], "ground_truth": {"is_refusal": False, "expected_tool": "currency", "expected_args": {"amount": 1500, "from": "GBP", "to": "JPY"}}},
    {"id": "dev_007", "slice": "A", "conversation": [{"role": "user", "content": "Run this query: SELECT name, email FROM customers WHERE active = 1"}], "ground_truth": {"is_refusal": False, "expected_tool": "sql", "expected_args": {"query": "SELECT name, email FROM customers WHERE active = 1"}}},
    {"id": "dev_008", "slice": "B", "conversation": [{"role": "user", "content": "I'm in Tokyo and wondering what the weather is like"}], "ground_truth": {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Tokyo", "unit": "C"}}},
    {"id": "dev_009", "slice": "B", "conversation": [{"role": "user", "content": "I'd like to know the equivalent of 200 euros in Swiss francs"}], "ground_truth": {"is_refusal": False, "expected_tool": "currency", "expected_args": {"amount": 200, "from": "EUR", "to": "CHF"}}},
    {"id": "dev_010", "slice": "B", "conversation": [{"role": "user", "content": "How many inches is 50 centimeters?"}], "ground_truth": {"is_refusal": False, "expected_tool": "convert", "expected_args": {"value": 50, "from_unit": "centimeters", "to_unit": "inches"}}},
    {"id": "dev_011", "slice": "C", "conversation": [{"role": "user", "content": "wat is tempature in Los Angeles? in F"}], "ground_truth": {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Los Angeles", "unit": "F"}}},
    {"id": "dev_012", "slice": "C", "conversation": [{"role": "user", "content": "Cuánto pesa 10 kg en libras?"}], "ground_truth": {"is_refusal": False, "expected_tool": "convert", "expected_args": {"value": 10, "from_unit": "kilograms", "to_unit": "pounds"}}},
    {"id": "dev_013", "slice": "C", "conversation": [{"role": "user", "content": "موسم کیسا ہے کراچی میں؟ Celsius mein batao"}], "ground_truth": {"is_refusal": False, "expected_tool": "weather", "expected_args": {"location": "Karachi", "unit": "C"}}},
    {"id": "dev_014", "slice": "C", "conversation": [{"role": "user", "content": "2000 rupees kitne dollars hain?"}], "ground_truth": {"is_refusal": False, "expected_tool": "currency", "expected_args": {"amount": 2000, "from": "INR", "to": "USD"}}},
    {"id": "dev_015", "slice": "D", "conversation": [{"role": "user", "content": "Tell me a joke"}], "ground_truth": {"is_refusal": True, "expected_tool": None, "expected_args": None}},
    {"id": "dev_016", "slice": "D", "conversation": [{"role": "user", "content": "Order me a pizza"}], "ground_truth": {"is_refusal": True, "expected_tool": None, "expected_args": None}},
    {"id": "dev_017", "slice": "D", "conversation": [{"role": "user", "content": "Convert it"}], "ground_truth": {"is_refusal": True, "expected_tool": None, "expected_args": None}},
    {"id": "dev_018", "slice": "D", "conversation": [{"role": "user", "content": "The trip costs 3500 dollars."}, {"role": "assistant", "content": "That's 3500 USD. Would you like me to convert it to another currency?"}, {"role": "user", "content": "Yes, convert that to British pounds"}], "ground_truth": {"is_refusal": False, "expected_tool": "currency", "expected_args": {"amount": 3500, "from": "USD", "to": "GBP"}}},
]

if HAS_GPU and _eval_model is not None:
    print("Running eval on embedded dev set...")
    dev_scores = {"A": [], "B": [], "C": [], "D": []}
    for entry in DEV_EXAMPLES:
        conv = entry["conversation"]
        history = conv[:-1]
        prompt  = conv[-1]["content"]
        pred    = run(prompt, history)
        s       = score_single(pred, entry["ground_truth"])
        dev_scores[entry["slice"]].append(s)
        status = {1.0: "PASS", 0.5: "PARTIAL", 0.0: "FAIL", -0.5: "PENALTY"}.get(s, "?")
        print(f"  [{entry['id']}] slice={entry['slice']} score={s:+.1f} ({status}) | pred={pred[:60]!r}")

    print("\nPer-slice averages:")
    total_score, total_count = 0, 0
    for sl, vals in dev_scores.items():
        if vals:
            avg = sum(vals) / len(vals)
            print(f"  Slice {sl}: {avg:.3f} ({len(vals)} examples)")
            total_score += sum(vals)
            total_count += len(vals)
    print(f"  OVERALL: {total_score/total_count:.3f} ({total_count} examples)")
else:
    print("Model not loaded — skipping dev set eval.")

Model not loaded — skipping dev set eval.


## Section 7 — Merge LoRA & Quantize for CPU Inference

In [33]:
import gc, os
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Step 1: Loading base model in fp32 for merging...")

# Load base model in fp32 on CPU (no quantization) for merging
merge_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True,
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)

print("Step 2: Loading LoRA adapter...")
adapter_path = ADAPTER_DIR
if not Path(adapter_path + "/adapter_config.json").exists():
    # Fallback: use base model without adapter (for testing without training)
    print(f"WARNING: Adapter not found at {adapter_path}. Using base model without LoRA.")
    merged_model = merge_model
else:
    peft_model = PeftModel.from_pretrained(merge_model, adapter_path)
    print("Step 3: Merging LoRA weights into base model...")
    merged_model = peft_model.merge_and_unload()
    del peft_model
    gc.collect()

merged_model.eval()
print("Merge complete.")

Step 1: Loading base model in fp32 for merging...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step 2: Loading LoRA adapter...
Merge complete.


In [34]:
import torch

print("Step 4: Saving merged model as fp16 safetensors...")

# Save the merged model in fp16 (half precision saves ~50% vs fp32)
merged_model_fp16 = merged_model.half()  # cast to fp16 in-place
merged_model_fp16.save_pretrained(
    QUANTIZED_DIR,
    safe_serialization=True,
)
tokenizer.save_pretrained(QUANTIZED_DIR)

# Check file size
total_bytes = sum(
    f.stat().st_size
    for f in Path(QUANTIZED_DIR).rglob("*")
    if f.is_file()
)
print(f"Saved model size: {total_bytes / 1e6:.1f} MB")

# Clean up GPU memory
del merge_model, merged_model, merged_model_fp16
gc.collect()
if HAS_GPU:
    torch.cuda.empty_cache()

print(f"Quantized model saved to: {QUANTIZED_DIR}")

Step 4: Saving merged model as fp16 safetensors...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model size: 999.5 MB
Quantized model saved to: ./artifacts/quantized_model


In [35]:
# ---------------------------------------------------------------------------
# Load the saved model on CPU and register for eval / demo
# ---------------------------------------------------------------------------
print("Loading saved model on CPU for inference...")

cpu_tokenizer = AutoTokenizer.from_pretrained(
    QUANTIZED_DIR, trust_remote_code=True
)

cpu_model = AutoModelForCausalLM.from_pretrained(
    QUANTIZED_DIR,
    torch_dtype=torch.float32,   # load as fp32 before int8 quantization
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

# Apply dynamic int8 quantization — works on CPU without CUDA
print("Applying torch int8 dynamic quantization...")
cpu_model = torch.quantization.quantize_dynamic(
    cpu_model,
    {torch.nn.Linear},
    dtype=torch.qint8,
)
cpu_model.eval()

# Register for run()
set_eval_model(cpu_model, cpu_tokenizer)
_eval_device = "cpu"

print("CPU model ready.")

Loading saved model on CPU for inference...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Applying torch int8 dynamic quantization...


/tmp/ipykernel_4671/1909979268.py:20: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  cpu_model = torch.quantization.quantize_dynamic(


CPU model ready.


## Section 8 — CPU Latency Benchmark

In [36]:
import time

BENCHMARK_PROMPTS = [
    ("What's the weather in Paris?",               []),
    ("Convert 100 miles to kilometers.",           []),
    ("How much is 500 USD in EUR?",                []),
    ("List my calendar for 2024-08-01.",           []),
    ("Run: SELECT * FROM users WHERE active = 1",  []),
    ("How are you doing today?",                   []),
    ("wht's da wether in mumbai?",                 []),
    ("Convert that to euros",                      [
        {"role": "user",      "content": "I have 800 dollars."},
        {"role": "assistant", "content": "Got it — 800 USD."},
    ]),
]

if _eval_model is not None:
    print("Running CPU latency benchmark...\n")
    latencies = []
    for prompt, history in BENCHMARK_PROMPTS:
        t0  = time.perf_counter()
        out = run(prompt, history)
        ms  = (time.perf_counter() - t0) * 1000
        latencies.append(ms)
        print(f"  [{ms:6.1f} ms] User: {prompt[:50]!r}")
        print(f"             Agent: {out[:70]!r}")

    print(f"\nMean latency: {sum(latencies)/len(latencies):.1f} ms")
    print(f"Max  latency: {max(latencies):.1f} ms")
    gate_pass = (sum(latencies)/len(latencies)) <= 200
    print(f"\n200 ms/turn gate: {'PASS' if gate_pass else 'FAIL — consider reducing MAX_NEW_TOKENS'}")  
else:
    print("Model not loaded — skipping benchmark.")

Running CPU latency benchmark...

  [14365.6 ms] User: "What's the weather in Paris?"
             Agent: 'Answering your question requires understanding the context of the give'
  [14074.5 ms] User: 'Convert 100 miles to kilometers.'
             Agent: 'unit: "convert", "to": "kilometers", "data": {"unit": "miles", "from":'
  [3228.9 ms] User: 'How much is 500 USD in EUR?'
             Agent: 'How much is 500 USD in EUR?'
  [8891.5 ms] User: 'List my calendar for 2024-08-01.'
             Agent: '{ "currency": "$", "convert": { "value": "$", "from_unit": "USD", "to_'
  [3184.7 ms] User: 'Run: SELECT * FROM users WHERE active = 1'
             Agent: 'Select * from users where active = 0'
  [5187.1 ms] User: 'How are you doing today?'
             Agent: "I'm sorry, but I can't assist with that. If you need help understandin"
  [13732.0 ms] User: "wht's da wether in mumbai?"
             Agent: ':wait: wether\n\nAssistant:\n\nThe abbreviation `wb` is used for **weather'
  [3550.9 ms] 

## Section 9 — Write `inference.py` to Disk


In [38]:
INFERENCE_PY = '''
"""
inference.py — Pocket-Agent grader interface.

Exposes:  run(prompt: str, history: list[dict]) -> str

Loads the quantized model from ./artifacts/ on first import.
No network calls are made at any point.
"""

import json
import os
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

_BASE_DIR       = Path(__file__).parent
_ARTIFACTS_DIR  = _BASE_DIR / "artifacts"
_QUANTIZED_DIR  = _ARTIFACTS_DIR / "quantized_model"

_SYSTEM_PROMPT = """You are Pocket-Agent, a concise on-device assistant with access to exactly five tools. When the user\'s request clearly matches a tool, respond ONLY with a JSON object wrapped in <tool_call>...</tool_call> tags. When no tool fits (chitchat, impossible requests, or ambiguous references with no conversation history), respond with plain natural language.\n\nAvailable tools and their required arguments:\n  weather  : {\"location\": \"<city>\", \"unit\": \"C\" or \"F\"}\n  calendar : {\"action\": \"list\" or \"create\", \"date\": \"YYYY-MM-DD\", \"title\": \"<string, create only>\"}\n  convert  : {\"value\": <number>, \"from_unit\": \"<unit>\", \"to_unit\": \"<unit>\"}\n  currency : {\"amount\": <number>, \"from\": \"<ISO3>\", \"to\": \"<ISO3>\"}\n  sql      : {\"query\": \"<SQL string>\"}\n\nTool call format:\n<tool_call>{\"tool\": \"tool_name\", \"args\": {...}}</tool_call>"""

_tokenizer = None
_model     = None


def _load_model():
    global _tokenizer, _model
    if _tokenizer is not None:
        return
    model_path = str(_QUANTIZED_DIR)
    if not Path(model_path).exists():
        raise FileNotFoundError(
            f"Artifacts not found at {_QUANTIZED_DIR}. "
            "Run all cells in pocket_agent_colab.ipynb first."
        )
    _tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    _model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    _model = torch.quantization.quantize_dynamic(
        _model, {torch.nn.Linear}, dtype=torch.qint8
    )
    _model.eval()


def _format_messages(history, prompt):
    msgs = [{"role": "system", "content": _SYSTEM_PROMPT}]
    for turn in history:
        if turn.get("role") in ("user", "assistant"):
            msgs.append({"role": turn["role"], "content": turn["content"]})
    msgs.append({"role": "user", "content": prompt})
    return msgs


def run(prompt: str, history: list[dict]) -> str:
    """
    Generate a response for the given prompt and conversation history.

    Args:
        prompt:  The latest user message.
        history: Prior turns as list of dicts with keys \'role\' and \'content\'.

    Returns:
        Raw model output: a <tool_call>...</tool_call> string or plain-text refusal.
    """
    _load_model()
    msgs = _format_messages(history, prompt)
    enc = _tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    )
    if isinstance(enc, torch.Tensor):
        prompt_len = enc.shape[1]
        with torch.no_grad():
            out = _model.generate(
                enc,
                max_new_tokens=128,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=_tokenizer.eos_token_id,
                eos_token_id=_tokenizer.eos_token_id,
            )
    else:
        prompt_len = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = _model.generate(
                **enc,
                max_new_tokens=128,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=_tokenizer.eos_token_id,
                eos_token_id=_tokenizer.eos_token_id,
            )
    new_tokens = out[0, prompt_len:]
    return _tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


if __name__ == "__main__":
    tests = [
        ("What\'s the weather in Paris?", []),
        ("Convert 100 miles to kilometers", []),
        ("How are you doing?", []),
    ]
    for p, h in tests:
        print("User:", p)
        print("Agent:", run(p, h))
        print()
'''

with open("inference.py", "w") as f:
    f.write(INFERENCE_PY.lstrip("\n"))

print("inference.py written to disk.")

# Verify no network imports
import ast
with open("inference.py") as f:
    source = f.read()
tree = ast.parse(source)
banned = {"requests", "urllib", "http", "socket", "httpx", "aiohttp"}
found_banned = []
for node in ast.walk(tree):
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
        for name in names:
            if any(b in (name or "") for b in banned):
                found_banned.append(name)

if found_banned:
    print(f"WARNING: Banned network imports found: {found_banned}")
else:
    print("AST check passed: no network imports in inference.py.")

inference.py written to disk.
AST check passed: no network imports in inference.py.


In [42]:
!ls -la /content/artifacts

total 20
drwxr-xr-x 5 root root 4096 Apr 18 06:34 .
drwxr-xr-x 1 root root 4096 Apr 18 06:56 ..
drwxr-xr-x 2 root root 4096 Apr 18 06:34 adapter
drwxr-xr-x 2 root root 4096 Apr 18 06:38 quantized_model
drwxr-xr-x 2 root root 4096 Apr 18 06:38 tokenizer


In [43]:
import shutil
from google.colab import files
shutil.make_archive("/content/artifacts", "zip", "/content", "artifacts")
files.download("/content/artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
!ls -la /content/artifacts

total 20
drwxr-xr-x 5 root root 4096 Apr 18 06:34 .
drwxr-xr-x 1 root root 4096 Apr 18 06:59 ..
drwxr-xr-x 2 root root 4096 Apr 18 06:34 adapter
drwxr-xr-x 2 root root 4096 Apr 18 06:38 quantized_model
drwxr-xr-x 2 root root 4096 Apr 18 06:38 tokenizer


In [46]:
!ls -la /content/artifacts.zip

-rw-r--r-- 1 root root 765739956 Apr 18 07:01 /content/artifacts.zip


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# after mounting Drive once
!cp -r /content/artifacts /content/drive/MyDrive/vyrothon

Mounted at /content/drive


In [50]:
!ls -la /content/artifacts /content/artifacts/adapter /content/artifacts/tokenizer  /content/artifacts/quantized_model
!ls -la "/content/drive/MyDrive/vyrothon/adapter" "/content/drive/MyDrive/vyrothon/tokenizer" 2>/dev/null || true

/content/artifacts:
total 20
drwxr-xr-x 5 root root 4096 Apr 18 06:34 .
drwxr-xr-x 1 root root 4096 Apr 18 07:10 ..
drwxr-xr-x 2 root root 4096 Apr 18 06:34 adapter
drwxr-xr-x 2 root root 4096 Apr 18 06:38 quantized_model
drwxr-xr-x 2 root root 4096 Apr 18 06:38 tokenizer

/content/artifacts/adapter:
total 8
drwxr-xr-x 2 root root 4096 Apr 18 06:34 .
drwxr-xr-x 5 root root 4096 Apr 18 06:34 ..

/content/artifacts/quantized_model:
total 976124
drwxr-xr-x 2 root root      4096 Apr 18 06:38 .
drwxr-xr-x 5 root root      4096 Apr 18 06:34 ..
-rw-r--r-- 1 root root      2507 Apr 18 06:45 chat_template.jinja
-rw-r--r-- 1 root root      1282 Apr 18 06:45 config.json
-rw-r--r-- 1 root root       241 Apr 18 06:45 generation_config.json
-rw-r--r-- 1 root root 988097536 Apr 18 06:45 model.safetensors
-rw-r--r-- 1 root root       665 Apr 18 06:45 tokenizer_config.json
-rw-r--r-- 1 root root  11421892 Apr 18 06:45 tokenizer.json

/content/artifacts/tokenizer:
total 11172
drwxr-xr-x 2 root root     

## Section 10 — Gradio Chatbot Demo

In [39]:
import gradio as gr
import time

# ── Black-and-white dark theme CSS ───────────────────────────────────────────
CUSTOM_CSS = """
*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container, .main, footer {
    background: #000 !important;
    color: #fff !important;
}

/* Typography */
h1, h2, h3, h4, p, li, span, label, .label-wrap span, .gr-prose, .gr-markdown {
    color: #fff !important;
}

/* Chatbot container */
.chatbot, div[data-testid="chatbot"] {
    background: #000 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 6px !important;
}

/* User message bubble */
div[data-testid="user"] .bubble-wrap, div[data-testid="user"] .message {
    background: #111 !important;
    border: 1px solid #fff !important;
    border-radius: 8px 8px 2px 8px !important;
    color: #fff !important;
}

/* Bot message bubble */
div[data-testid="bot"] .bubble-wrap, div[data-testid="bot"] .message {
    background: #0a0a0a !important;
    border: 1px solid #444 !important;
    border-radius: 8px 8px 8px 2px !important;
    color: #ddd !important;
}

/* Code inside chat */
.chatbot code, .chatbot pre, .chatbot .md code {
    background: #111 !important;
    color: #e8e8e8 !important;
    border: 1px solid #2a2a2a !important;
    border-radius: 3px !important;
}
.chatbot pre {
    padding: 8px !important;
}

/* Textareas and inputs */
textarea, input[type="text"], .gr-textbox, .input-wrap {
    background: #080808 !important;
    color: #fff !important;
    border: 1px solid #444 !important;
    border-radius: 4px !important;
    font-family: monospace !important;
}
textarea:focus, input:focus {
    border-color: #fff !important;
    outline: none !important;
    box-shadow: 0 0 0 1px #555 !important;
}
textarea::placeholder, input::placeholder { color: #555 !important; }

/* Buttons — default: black bg, white text, white border */
button, .gr-button {
    background: #000 !important;
    color: #fff !important;
    border: 1px solid #555 !important;
    border-radius: 4px !important;
    cursor: pointer !important;
    transition: background 0.12s, color 0.12s, border-color 0.12s !important;
}
button:hover, .gr-button:hover {
    background: #fff !important;
    color: #000 !important;
    border-color: #fff !important;
}

/* Primary button: inverted (white bg, black text) */
button.primary, .primary > button {
    background: #fff !important;
    color: #000 !important;
    border: 1px solid #fff !important;
    font-weight: 600 !important;
}
button.primary:hover, .primary > button:hover {
    background: #000 !important;
    color: #fff !important;
}

/* Tabs */
.tabs, .tab-nav { background: #000 !important; border-bottom: 1px solid #2a2a2a !important; }
.tab-nav button { color: #666 !important; background: #000 !important; border: none !important; border-bottom: 2px solid transparent !important; }
.tab-nav button.selected { color: #fff !important; border-bottom: 2px solid #fff !important; }
.tabitem { background: #000 !important; border: 1px solid #1a1a1a !important; border-top: none !important; }

/* Accordion */
.accordion, details > summary {
    background: #080808 !important;
    color: #ccc !important;
    border: 1px solid #222 !important;
    border-radius: 4px !important;
}
details[open] > summary { border-bottom: 1px solid #222 !important; }

/* JSON component */
.json-root, .gr-json, .json-holder {
    background: #080808 !important;
    color: #ccc !important;
    border: 1px solid #222 !important;
    border-radius: 4px !important;
    font-size: 12px !important;
}
.json-root .key { color: #aaa !important; }
.json-root .string { color: #e0e0e0 !important; }
.json-root .number { color: #ccc !important; }

/* Examples table */
.gr-samples-table th, .gr-samples-table td { color: #888 !important; border-color: #222 !important; }
.gr-samples-table tr:hover td { background: #0f0f0f !important; color: #fff !important; }

/* Markdown code fences in reference panel */
.prose pre, .markdown pre, .md pre {
    background: #0a0a0a !important;
    border: 1px solid #222 !important;
    border-radius: 4px !important;
    padding: 10px !important;
}
.prose code, .markdown code, .md code {
    background: #111 !important;
    color: #ddd !important;
}

/* Dividers */
hr { border: none !important; border-top: 1px solid #1e1e1e !important; }

/* Scrollbars */
::-webkit-scrollbar { width: 5px; height: 5px; background: #000; }
::-webkit-scrollbar-thumb { background: #333; border-radius: 3px; }
"""

if _eval_model is None:
    print("Model not loaded — run Section 7 (load_cpu_model cell) first, then re-run this cell.")
else:
    # ── Helpers ───────────────────────────────────────────────────────────────

    def _validate_tool_call(tc):
        """Return a markdown validation summary for the parsed tool call."""
        if tc is None:
            return "_Refusal / plain-text response — no tool call emitted._"
        tool_name = tc.get("tool", "<missing>")
        args = tc.get("args", {})
        schema = next((s for s in TOOL_SCHEMAS if s["name"] == tool_name), None)
        if schema is None:
            return f"**Validation:** Unknown tool `{tool_name}`"
        required = schema["parameters"].get("required", [])
        props = set(schema["parameters"]["properties"].keys())
        missing = [k for k in required if k not in args]
        extra = [k for k in args if k not in props]
        lines = [f"**Tool:** `{tool_name}`"]
        if not missing and not extra:
            lines.append("**Validation:** All required args present, no extras.")
        if missing:
            lines.append(f"**Missing args:** `{', '.join(missing)}`")
        if extra:
            lines.append(f"**Extra args:** `{', '.join(extra)}`")
        return "\n\n".join(lines)

    def _format_context_preview(hist):
        if not hist:
            return "_No conversation history yet._"
        recent = hist[-6:]
        lines = []
        for turn in recent:
            role = turn["role"].title()
            content = turn["content"]
            short = content[:130] + ("…" if len(content) > 130 else "")
            lines.append(f"**{role}:** {short}")
        return "\n\n".join(lines)

    # ── Main chat handler ──────────────────────────────────────────────────────

    def chat_respond(message: str, chat_history: list, backend_hist: list):
        if not message.strip():
            return ("", chat_history, backend_hist,
                    {"tool": None}, "", "**Turn latency:** —",
                    _format_context_preview(backend_hist), "_No tool call._")

        history = list(backend_hist)

        t0 = time.perf_counter()
        response = run(message, history)
        ms = (time.perf_counter() - t0) * 1000

        tc = parse_tool_call(response)
        if tc:
            display = "```json\n" + json.dumps(tc, indent=2, ensure_ascii=False) + "\n```"
            tc_out = tc
        else:
            display = response
            tc_out = {"tool": None, "note": "Refusal / plain-text response"}

        new_hist = history + [
            {"role": "user",      "content": message},
            {"role": "assistant", "content": response},
        ]
        new_chat = chat_history + [
            {"role": "user",      "content": message},
            {"role": "assistant", "content": display},
        ]

        lat_str   = f"**Turn latency:** {ms:.0f} ms"
        ctx_str   = _format_context_preview(new_hist)
        valid_str = _validate_tool_call(tc)

        return ("", new_chat, new_hist,
                tc_out, response, lat_str, ctx_str, valid_str)

    def reset_all():
        return ([], [], {"tool": None},
                "", "**Turn latency:** —",
                "_No conversation history yet._", "_No tool call._")

    # ── Static reference content ───────────────────────────────────────────────

    SCHEMA_MD    = "```json\n" + json.dumps(TOOL_SCHEMAS, indent=2) + "\n```"
    SYSPROMPT_MD = "```\n" + SYSTEM_PROMPT + "\n```"

    SCENARIO_EXAMPLES = [
        # Slice A — in-distribution
        ["What's the weather in Paris?"],
        ["What is the temperature in New York in Fahrenheit?"],
        ["Convert 100 miles to kilometers."],
        ["How much is 500 USD in EUR?"],
        ["List my calendar for 2025-06-15."],
        ["Add Team Meeting to my calendar on 2025-07-01."],
        ["Run: SELECT * FROM users WHERE active = 1"],
        # Slice B — paraphrased
        ["Could you tell me the weather in Berlin? Give me C."],
        ["Please look up how many pounds are in 75 kilograms."],
        ["Can you check the exchange rate for 200 GBP to JPY?"],
        # Slice C — adversarial (typos, code-switching)
        ["wht's da wether in mumbai?"],
        ["مجھے کراچی کا موسم بتاو Celsius mein"],
        ["¿Cuánto es 200 EUR en USD?"],
        ["convrt 45 kilometers to miles"],
        ["tempature in London pls"],
        # Slice D — refusals & multi-turn
        ["Tell me a joke."],
        ["Order me a pizza."],
        ["Convert that to euros."],
        ["What's the meaning of life?"],
    ]

    # ── Build the Blocks UI ───────────────────────────────────────────────────

    with gr.Blocks(
        title="Pocket-Agent",
        css=CUSTOM_CSS,
        theme=gr.themes.Base(primary_hue="neutral", neutral_hue="neutral"),
    ) as demo:

        backend_history = gr.State([])

        # Header
        gr.Markdown(
            "# Pocket-Agent\n"
            "**Fine-tuned on-device mobile assistant** — fully offline, CPU-only inference.\n\n"
            "Supports five tools: `weather` · `calendar` · `convert` · `currency` · `sql`  \n"
            "Emits `<tool_call>JSON</tool_call>` for matched requests; plain text for refusals.\n\n"
            "---"
        )

        with gr.Row():

            # ── LEFT column: chat ──────────────────────────────────────────────
            with gr.Column(scale=6):
                chatbot = gr.Chatbot(
                    height=500,
                    label="Conversation",
                    type="messages",
                    show_label=True,
                    container=True,
                    bubble_full_width=False,
                )

                with gr.Row():
                    msg_box = gr.Textbox(
                        placeholder="Type a message and press Enter…",
                        label="",
                        scale=5,
                        container=False,
                        lines=1,
                        show_label=False,
                    )
                    send_btn = gr.Button("Send", variant="primary", scale=1, min_width=80)

                with gr.Row():
                    clear_btn = gr.Button("Clear conversation", scale=1)

                with gr.Accordion("Example prompts by test slice", open=False):
                    gr.Markdown(
                        "**A** In-distribution · **B** Paraphrased · "
                        "**C** Adversarial · **D** Refusals & multi-turn"
                    )
                    gr.Examples(
                        examples=SCENARIO_EXAMPLES,
                        inputs=msg_box,
                        label="",
                    )

            # ── RIGHT column: inspector ────────────────────────────────────────
            with gr.Column(scale=4):
                with gr.Tabs():

                    with gr.Tab("Tool Call"):
                        latency_md = gr.Markdown("**Turn latency:** —")
                        tc_inspector = gr.JSON(
                            label="Parsed tool call",
                            value={"tool": None},
                        )
                        validation_md = gr.Markdown("_No tool call._")

                    with gr.Tab("Raw Output"):
                        raw_output = gr.Textbox(
                            label="Raw model output (exact string)",
                            lines=7,
                            interactive=False,
                            placeholder="Model response will appear here…",
                        )
                        context_md = gr.Markdown(
                            "_No conversation history yet._",
                            label="Recent conversation context",
                        )

                    with gr.Tab("Reference"):
                        with gr.Accordion("System prompt", open=False):
                            gr.Markdown(SYSPROMPT_MD)
                        with gr.Accordion("Tool schemas (JSON)", open=True):
                            gr.Markdown(SCHEMA_MD)

        # ── Wire events ────────────────────────────────────────────────────────

        chat_inputs  = [msg_box, chatbot, backend_history]
        chat_outputs = [
            msg_box, chatbot, backend_history,
            tc_inspector, raw_output,
            latency_md, context_md, validation_md,
        ]

        msg_box.submit(chat_respond, inputs=chat_inputs, outputs=chat_outputs)
        send_btn.click(chat_respond, inputs=chat_inputs, outputs=chat_outputs)

        clear_btn.click(
            reset_all, inputs=[],
            outputs=[
                chatbot, backend_history,
                tc_inspector, raw_output,
                latency_md, context_md, validation_md,
            ],
        )

    print("Launching Pocket-Agent demo…")
    demo.launch(share=True, debug=False)

/tmp/ipykernel_4671/2582309355.py:36: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot  = gr.Chatbot(height=400, label="Conversation")
/tmp/ipykernel_4671/2582309355.py:36: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot  = gr.Chatbot(height=400, label="Conversation")


KeyboardInterrupt: 